# **ESMFold Batch Processing**
Fold 200+ protein sequences in parallel. For details see: [Github](https://github.com/facebookresearch/esm/tree/main/esm)

## **Instructions**
1. Run the **Install** cell first (one-time, ~3 min)
2. Paste your sequences in **Batch Input** cell (FASTA format)
3. Run **Fold** cell (~1-2 min per 100 sequences on T4 GPU)
4. Run **Download** cell to get all PDBs as a zip

## **Tips**
- Max sequence length: ~900 aa (limited by T4 GPU memory)
- Sequences are folded in order, one at a time
- Output includes PDB files + summary CSV with pTM/pLDDT scores

In [1]:
%%time
#@title **Install ESMFold** (Run once)
#@markdown This will download ESMFold model weights (~2GB) and install dependencies (~3 min)

version = "1" # @param ["0", "1"]
model_name = "esmfold_v0.model" if version == "0" else "esmfold.model"
import os, time

if not os.path.isfile(model_name):
  # download esmfold params
  os.system("apt-get install aria2 -qq")
  os.system(f"aria2c -q -x 16 https://colabfold.steineggerlab.workers.dev/esm/{model_name} &")

  if not os.path.isfile("finished_install"):
    # install libs
    print("installing libs...")
    os.system("pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol modelcif")
    os.system("pip install -q git+https://github.com/NVIDIA/dllogger.git")

    print("installing openfold...")
    # install openfold
    os.system(f"pip install -q git+https://github.com/sokrypton/openfold.git")

    print("installing esmfold...")
    # install esmfold
    os.system(f"pip install -q git+https://github.com/sokrypton/esm.git")
    os.system("touch finished_install")

  # wait for Params to finish downloading...
  if os.path.isfile(f"{model_name}.aria2"):
    print("downloading params...")
  while os.path.isfile(f"{model_name}.aria2"):
    time.sleep(5)

print("Installation complete!")

installing libs...
installing openfold...
installing esmfold...
Installation complete!
CPU times: user 12.7 ms, sys: 4.96 ms, total: 17.6 ms
Wall time: 3min 30s


## Batch Input
Paste your sequences below in FASTA format. Example:
```
>sequence_1
MVAHKLTLEVQGVIQKVSQQQTSGHAK
>sequence_2
MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEK...
```

In [2]:
#@title **Paste your FASTA sequences here**
fasta_input = """
>seq_0
PSVVEVPKGVLRVFDDLLVTVPANRDLRVIAHQNTEVLTKRLLAADYTVQHRIVMLAPGEASTLPLRQLRPGKYVLHAQNIQTANQSRAGILVQE
>seq_1
GTLYLQTEKTVFKQDEVLHLRALIRGEERDWKGVTLRILNDKGELVKKLTSSPFRFNAAEFDIRVRHRQGNYRVVIEMTDVDGSKLTRELTIEVE
>seq_2
QQIVTVSGGVLAHDPASNITIPRNHALQVENKDAVPWVLRHTDMHSRRVVAGPVTLTTSQTKTWIDLKPLEPGQYQLQATNSNGSVQICNLTVVS
>seq_3
MGFSPASVTLAAGETATLTLVLDDDAVVHGFTLLKDGTPIAPATDGSGFQLLDDGRKVRLTSTTSGTGLTATLTLVLHTADEERTLTATVTGVAP
>seq_4
ENVTVTLDQTSYDIGTTVTVQISLSRAPSSPMIVRLGVLDSHRKLVHAKEQSADQQASLTLKLTYTGDAGNYTLSVSISSPSGKKASSQLVFQVK
>seq_5
AMFHVVTIQQGTLSLSPGNIISQSGRLLVKNMDEHDYLVTLFLYVGQIPKRVDVGEELQPGHATNPIPKWNLPSGEYKLVAVNKENKAVLSIRIQ
>seq_6
NVVLQCKSGKLLFGEDQPLRVENSHPLKVNNINNISQLDFLSLISNNVNQNEAKNVYPNEYVEFDLTRLRPGEYRIHCREVDSGKRHLCTLEVTQ
>seq_7
MRLVLKLERSVYRKNEYVYFSVSLEGNWRNWRELIFLVKDKAGNVLHEQKFSPFETQHFFHRYKANFEDGDYDVQVEATWQDFRYVLAHADIRVN
>seq_8
NVEIQCKNGKLLKGDAQPVRVENSHPLKVNTFNYKSQLDFLSLISNNVNQNEAKNVYPNEYKEFDLTRIRPGEYRIHCIEVDSGKRHLCTLEVYE
>seq_9
TTEVKVVVRNGVLELAEGAELEPGETLSITNENPVMDDIWVYREEGENENLLPGWSHVYPNRTYTTVLPSTIKRGIYKVVSQSAKFADSVEFNVP
>seq_10
KSTAIVEVNDGKVTIISGEVLDSQGEIVIENKSKAVINVKVWFAMGGMVQLVDMLTGIKPGQVKTSKITVNVTGGILEITCTHNEGNGRTMILIN
>seq_11
THTIEVQVRNNKAVCTGPGELKVKVGETEVVNVTNASMDKELVAELYDARGNSLATVKAGAGAPTLSLRIDDPGLTDFRITITLTDGHTCTVTLEVVVGT
>seq_12
APAERFLSLTLDEKTYRQGDIVTISSENLSRTFDLTINIFKHSSNGRLVEVLVKNDIVSPLSRTAKQFTVEEPAGIFIFQAIARRGSRACRFLLEIINVN
>seq_13
SNVITVVIKNGILEEIKGTNIGPGEVVNVTNSDYEPHTFVVELNEQDKKITVKTLQDVPPGMAKEWVAEENEQTGLYIITANGATHQSSVAFSLS
>seq_14
YCVIPLLQKNIYQRGEDVLVRIQFSEAHNYPYTVKLILLDSEGYVVHVHTERDKKEKMLNLNYRVEELPYDEFLRALVTSKNGQVLEAKAFLRVE
>seq_15
GLLTLQLDKSNYKTGTVAYLRIVMAYRLFSDGAVRFSVKDYFGRELKLDELQLRRVTEIMKEFTVNFSIGVYKVEATLLEPDDSRRSASSSITVE
>seq_16
TDEAKVVVRNGVYEIAEGAQLGREGTLSITNENPVADRIWVGRVGPRGENEDIGSFIGYPNRTRTTVLRSTMGGEQLVVASQSAKFASAATINVP
>seq_17
PTLAVIVVKDHRAFLEKGDEVDPGGIVKFRNDAEETVTIEVNLNVDGIEILIHNYSQVMPGQEVVINIPKDLPSGKYWIMTTTAKYRDWVVLELT
>seq_18
GCVIPLLDKNIYQRGEDVLVRIQFSEAHNYPATVKLILLDSEGYVVHVRTESDKKEKMLNLNYRVEELPYDYFLRALVTYPNGQVLEAKAFLRVE
>seq_19
TDEAKVVVRNGVYEVAEGTELGREGTLSITNENPVPDTIWIYREEGENTNLLKTISHMYPNRTYTTVLRLTMGGGILKVVSQSAKGAGSATINIP
>seq_20
GLLTLQLDKSNYKTGTVAYLRIVMDYRLFSDGAVRFSVKDYFGRELKLDELQLRRVTEIMKEFTVNFSIGVYKVEATLLEPDDSRRSASSSITVE
>seq_21
RLLFTVSGANGELSIEPHNTIPLDGLIQFSNLGQSPLDFFLLYSDGGNSYECLSIENLPPGRTYRQFDTRAIPAGRYQIAMYTIEFEAQVDITIK
>seq_22
TDEVKVVVRNSVYEVAEGTELGREGKLSITNENPVMDTIWIYREEGENTNLLKTLSHVYPNRTYTTVLRLTMGGGILKVVSQSAKGAGSATINIP
>seq_23
MRPVYKHKLVVDVWHGAEYVSKGSVDIWLSTRELGEARVEIISENGVFIYDTFFEIEHFFHREQADLTDIDPGRYKVRAWTTFDYNYRHFDVRPN
>seq_24
AVCLRLSATRKKFPRGEIMLLVSADERSNFTLTISLENMNKTVIGTLYRGTSIFTEFYLPFDTSDLEDGWYKISAWLGWKQANHSIDIVQ
>seq_25
GLLTLQLDKSNYKTGTVAYLRIVMDYRLFSDGAVRFSVKDYFGREVKLDELQLRRVTEIMKEFTVNFSIGVYKVEATLLEPDDSRRSASSSITVE
>seq_26
NVEIQAKNGKLLAGDAQPVRVENSHPLKVNTFNIGSQLDFLSLISNNVNQNEAKNVYPNEYVEFDLTRIRPGEYRIHCIEVDSGKRHLHTLKVTE
>seq_27
ATIELRTRSFHYAKGQTVNHSISLEESAREDRLITLEVLDEDGTVVKKWKTFSNNRREVVFRSVLEFEGGRYLLTAFVTTLDGGVWEANTKIDVK
>seq_28
SEEITVVIKNGQVEEIKGTNVGKDFVLNLTNSDYEPHTFVVEANEQDKKITVKTLQDVRPGMAKEWSAELNEDTKLDIKAFNGDGHVSSVAISVS
>seq_29
TEQAWILRVGTDKAEYHAATTNYVRYWAPNETQLTIRLLEKDWNLKKKIKEEQWPPGFNTTTERISFVNLKPGDYYILALLSTADERYAEFRWSG
>seq_30
VPETRPRLKDISIGPGQGVNIDQPVQVQASITKRDELDLRLIGRDRSISVLLRELRHGSSVSTLFPLRPLTPGDYIVQAFTDWVEGYRPLHTFQP
>seq_31
RQPLYTQRLVVLVNNGMPYVFKGDVLVKVCHKEQGTVRYEVLMDNNNKVMDEMITLRQDCLDYTFDTTNNQIGDYTFRAATDSLEAESQFTLKEQ
>seq_32
ATIELRTRSFHYAKGQTVNHSISLEESAREDRLITLEVLDADGTVVKKWKTFSNNRREVVFRSVLEFEGGRYLLTAFVTTLDGGVWEANTKIDVK
>seq_33
RLVKELKIDQWQEFPETAEITAIRIEVTDDDAENLEVKSAKVNEIPVSLVEDGSTLILDLRDNPLAVDKGDILRISIEFSAYKMYTVGSRIQTHRADGSRAGYDLHWTIG
>seq_34
PPLVIVADNVLGVPDASKITMNGDSSLDVENKDNEAVTLELVDSSSGKVVTGPFGLRAAGDKQGFDLTAGPEGKYYLWSRSHPKRQEKCNITVTK
>seq_35
ENVTVTLDQTSYDIGTTVTVQISLSRAPSSPMIVRLGVLDSHRGLVHATEQSADQQASLTLKLTYTAPAGNYTLSVSISSPSGKKASSQLVFQVK
>seq_36
RFVTLQLDRNYYLKNQDVLIKVIVTGENHRWKAVEIKMLSSDGTLVKQLNFNEFRDQKKTVRLTVNEKPGIMTIQVQLVDENEKSFEARHTVFIG
>seq_37
MTEVHIEVRGGEASITSGDSITPGGNIIINNSDDCVVNRSLLLKEDESAKQIQSWQDIEPGTLKVYSLPEDLPPGTYVAEVENETFWNKVYFHVE
>seq_38
MDEIEVVIKDGELSIRKGTRAGLKAKVRITNQSKSKVHVCVTMIAPGGEEAELYIGFLAAGETYELDVDFERPGCKLHIHGSSQEFSSTKTIEVL
>seq_39
AAVHSVTLAEGLLSLDPGTAIDLTDKLVIKNESFKVHHFNEHLVAGDKKIVVREIKRLFPGSSTPDFDLSKLQPGSYYIEATNQDFSASRKVTLQ
>seq_40
DCLTLLLEYETYAIGQELKLKDNLSQATQQDSLNTLQILDIKGRVLQQTEGMLQGQQTLTVVMTIQLPSGYRTFKSRVSYRDGSEFFSSSKFKVE
>seq_41
MVVCTFRDGKLGFPGHDAVNINNFTPLIIRNESEERITLQLRNVENECEVLGPFDKGPSGTLKETNLGRVPDGQYELYGTFRDKRKASCKIIVTQ
>seq_42
ALTEIKVYQVKVGDSLVDIQAWKEGNFLHLMITTATMKKPALITIIDNSNKVLMKTDFAPGGSYSTVIPDLGHSLLLKVSDGYNEISVIN
>seq_43
SSTVTVVIADGQVTITAGSALDPGDSLVLTNLGTAPTTFSLCHQVSKEEVELQTHTAVAPGQTVTIAIPKETPPGKYWLRSQNSTFECTATFELR
>seq_44
GLHEQAQNNNLTVRLKNGSEVTNEVLVEVTLAQDTPLLELSIINDNGEVVFTQKIENVPKNITVTVDVSWEPEGEYTVTVANEQGTKVNIQFTVK
>seq_45
SNSVTLVSKKGKLMIVSGKEVSPGGESTVYNKSYVPLNIALKRELLDEEIELDIKKQVDPAKQYKFRVPENVRDGLYKIVVISNKFNDTKDFRIS
>seq_46
RFVAEVTGNNLSVGTQKSLQIKIEKVLNIRTNNYPQVLDFLLLDRNGQVVRHERRVQPSQVAKIPLRDIPIMTYQLQAIDKLEGDEKKMHVVFIG
>seq_47
ATIELRTRSFHYAKGQTVNHSISLEESAREDRLITLEVLDEDGTVVKKWKTFSNRRREVVFRSVLEFEGGRYLLTAFVTTLDGGVWEANTKIDVK
>seq_48
DDVVYLQLSGGAVALVSKFKLEAGDVLKVINNDFNPWDLRILKETFDRAVELAKKENVNPNLTVSISDPKSLLPGRYTVQAVSGGFTDEADILLE
>seq_49
TSLQVTLDRSTYNLGEQVEVSIEFSSAESSDANLVLTVLSSEGEPILKLEQISQGKTDVQIFFTVEYQIGKYTLQVDRKDPTGKTEPASVTFEVK
>seq_50
SMTLQIDLEASAAGQKIQQGELIEYTETVISSEPVQLDLFIQNESNETLDLLEQTTREANQAKTIKKSANFKSGYYKLSLKTDNKVEDKPFWVTD
>seq_51
QSLQVTLDRSTYNLGEQVEVSIEFSSEESSDANLVLTVKSSEGEPILKLEQISQGKTDVQIFFTVEYQIGKYTLTVDRKDPTGKTEPASVTFEIK
>seq_52
GPESRVLIVDGKLSIVQGKQLPGTGNLVISNRSDKPYDIEVAHIAPSGKKEMVRSFTLQPGESRRIDLFFKEKGGTLKIIAATEDHIDEALVKVE
>seq_53
RLLPFAVQAFTVRPPEVHNRAFLTGSASGSTGQISIEVRDQLTSDALHSYSCTMESPGSLVRTYDFSPYRAGPYEARLRDEYAIEVAQQTFRVLR
>seq_54
DLHITVTDGIVEVTPEVQIPQNTYVRITIENKDGESLYVVITNTDTESVKINEEVGTGTKNYDLLAEQIGTIPLKVTFIAFDGTKNTATTTLTVS
>seq_55
DCLTLLLEYETYAIGQELKLKDNLSKATQQESLNTLQILDIKGRVMQQTEGYSFGRQTMTVVMTIQLPSGYRTFKSRVSYRDGSEFFSSSKFFVV
>seq_56
PKTVVIVVCNGRLRLIEGRELRPGGTVNIRNEDKRTQTVTIEHVVQATRVVLQTLNSVEPSSSYDVPLPAHTPEGQYIVCAESTQFRDETLFQVA
>seq_57
SNLTLNLSKKEYFQGESATVTVKYSESTSYDVKYVFQVYSSQGELLREYQADADTKQVLVAKFTVQYNKGNYRLYATAIFNDGNKLTATKDFRIS
>seq_58
ASTELVSFQNGHLTILSGSSMSPGETLQVNNNNANEPDFKIHFKDGAAWNTLQKCTNLKPGQTRLFVLPEDLPSGTYTIIVESAVFKDRVEVALK
>seq_59
ATIELHLDRDAYEQGEMVELTVVFSAAASSDVVLNLRCLDYKGVMWFGSEWKAQKQIKQVYTFTINLPNGHYLLTAWFTEPNGKQLEASVRFTIY
>seq_60
SAISLVADRRIYVQGESAQLHVILEEAPVEPALVRITVLDRQDRECKKMTSMMNKSRSCPWNFTPFFDKGYYHVAVQVTTPSKETLQDCRLIIIK
>seq_61
TTVSKVVVRNSVLYLAPGTQLPREGYVSFTNENPVMDTFWMYRKEGENWNLIPGWSHMYPNRTTPTIDVSTMKRGPYKFVSQSAKFAASLELNVP
>seq_62
MRLVLKLERSVYRKGEYVTFTVSASMPWREDTELIFTVKDKAGNVLHEQSFTAQQKKHFTVRVKANFEDGDYDVQAEATWPDFRYVLAHADVRVN
>seq_63
DELVEATIRNGAVDLTPGEVIDQNSYLEFSNQSDVATNIELYLKKGLKWECIKTWKDIAPASATQEYQMTKIKPGEYRVIATTTTHPESVSFEIK
>seq_64
KATVNVTEGHVLFGQNSRLSVNHHYSFFVQTKGHPNTITVELEDEQGLVVKSIRTQQPKYQSKLDLKEMEVGKCLLTVKDADTSSSHVSEVFLFW
>seq_65
SQTIVNLESWNSVQPETKMEVSAKEPVEVINLDPVDLVLKTWDDETGAVVLGPFHLGRAEGSKSIDMTPEEAGAYTLAATTDDGTKIHFDLTITC
>seq_66
SVSVTVESGALTSTPTVKVKKGKYVTFSFSNKEGSAVHFIVEDVETGAGLINQTLQPGTWVQKFMAKDSGVKTYQFKVIFPAGSAAEKKMVLYVE
>seq_67
SQIKVSLDQLDYEQNQTACVLVSLTESPLEPVKLRFQVLDEDEMVVFSQCRDTKTNKDLRMEFQVIQPSGVYKIRATHVAQDGIKLDAEVRVTIH
>seq_68
DDIALTTCRDSYREGDTAWIEVSFVSPRPNTTELTLRVKNEAGELEFAKQHYAGKRARYQWQWGVHLPPGRYTVVVTATDINERKWEAEVHIQVL
>seq_69
QSLQVTLDRSTYNLGEQVEVSIEFSSAESSDANLVLTVLSSEGEPILKLEQISQGKTDVQIFFTVEYQIGKYTLQVDRKDPTGKTEPASVTFEIK
>seq_70
NKDVTIEIGRVDLADKAVSIKPGIMKVNIENNEGEMHQVCVINLDTVDMLINKYHITGNHEFLLNLDKKGTYAIEFVVTFMHNEVNKGTFSVTVV
>seq_71
AWVVRLTAGELGVDGAADLTVSAAAPLQLENHSPLSATIRTVETESYATVSGPVLLRAAGTKKTLNTEHLYEGHYLIEATFSAGMTAKARIHVII
>seq_72
RPIQLPQREQISLEPGMTVQRGEPTLVKEKNTAPAALRQEVLMKNLNTQMTLMPAVPQDSLDQTVDWYPNQIGDYHFYAADDSGEMAAQVVVKET
>seq_73
SLLTLTLDEDPFQRGRKANIHLFFGSPTLSIQKTIIRVLSTTGDLLKTDTYSAFRATSFSHNHVMEWDRGTYLVEANVTLSSNENFLAETKVDVW
>seq_74
WMCEVACSNTVYKLGSEVTISVDDETPPDEECLVALEVLDKNFQTQAYLEFKSDRKHEKNVSFVVTCPPGMYYIRCYVTFNNGKVKEAVVKIIVE
>seq_75
LGWTPASVTIGPNEEKEITLVLDSDLKILGFDLLKDGTVVAPSTDGTGFSLLDNGKKLKIKSTTSGTGLTATLTAVLKTETADLTATATVTGVEA
>seq_76
AQHTNRGDTLLTTPLRTVQLTPNDKVFVDLGPGYSPAFIMVLDRDETIIAQTLNQNVAANFTHNVDLRDFGEGIRKLVTGSPFTSRCRTFAVTSK
>seq_77
TDDTTTVYLTPQKATQGNQVSKMSLPEPGEYSLTLNGTQATPQRGTGAGPTMAVNGSGVTFRQLQWGQNSDAVIFEAKYKTTTNTTTQAKKERAE
>seq_78
KDTVKVSVENGIASITKGKVIHPGGHLLINNKSNEPVHIAKFLEVDDRLIRLKVFEELKPNDTVLLMVPKWVPPGKYRFTAENNYYKCEATFWVM
>seq_79
HKFDLRLDKEHYKKGSDVTLRVSALRPRSSNKKVSLNILDKDCVPLYKREKLANKLQSLVLTFKVEIDPGEVTVQQMETGDDGKIANAVKQITIN
>seq_80
NTIAEVVLAKGILRITQGTELVPGGELRITNEDDVNVPIGIYHQKENESVLLQQIEQLHKGEVVSVPLPLQATPGMISVTARDYNFYATATFRNG
>seq_81
ADFVVVNVQQGTLSLTQGEIASLSFRLLVKNMDEHDYLVTIFEYVGQIPKRLDVGEELQPGHTINVMIKMNEPSKLYIEVTTKDGQKDVVSIRVQ
>seq_82
GCVIPLLQKNIYQRGEDVLVRIQFSEAHNYPATVKLILLDSEGYVVHVHTERDKKEKMLNLNYRVEELPYDYFLRALVTYPNGQVLEAKAFLRVE
>seq_83
DHLNGLLDSTHFVIGNDVRCTIQASELQKNNSLLILQGLSLEQRLLWQEQKLLYAQKNLNFYFQVDFEWGTYIVVKTQTEANGQGFTASQVIQIR
>seq_84
PKTLVVRNGTLAYGLAKTLVVDRNTPVRIVNAEPSLVKAFRLKDASGCVIKSWPKIGPNTSVALSRLNTLKDGSSVVEAKGSDGTKTSFNLTVTE
>seq_85
DCLTLLTEKETYAIGQELKLKDNVSGNTQQWSKNTLEILDIKGRVLQQLETPSFESQTKTVVIRIDLPSGYRTFKSRVKKKDGSEFFSSIKVKVE
>seq_86
IPEFTLLKGQLTPKSNQIKISPGTINVKVSNRDGEPAIFELADVNTNQPLFMGRIDPGTAVFNLDIDEFNKFNYKFLVTLPDGSKHETEFLVEVE
>seq_87
WMCEVACSNTVYKLGSEVTISVDDETPPDEECLVALEVLDKNFQTQAYLEFKSDRVEEKNVSFVVTCPPGMYYFRCYVTFNNGKVKEAVVKLIVE
>seq_88
KNVFTAEVSAGTSNLTPGVLISKDDTFRFLNKSTEAVDFELLIKENDIGRIVYSEQKVAPNQETMEKDMPNLPPGDYVIKSASSSFRNALSFYVS
>seq_89
LSTAVVRDRRIGFNPDKRMEINAADLLRVQNDTACEMELEVTDSSTGYKYRGQYILGEAGEEKDIDLNEMEEGAYKLIARFKDFSKAALDIRVLV
>seq_90
MVVQTKTNLESVGTQTVEIEIWEDGIKIRFSIKAVTSQEPLKVKIIDDNGNIEKETKLPPMAEFNFELKNAGENYKVQITNQIHRTEYTF
>seq_91
SETVEISEGQLGFDPSQVRTISAASNLTVLNKSSDTHTVQLVDTQTKVAVLGPYTLGTKGSKKQIPLTLLPSGLYELKATCSSGHVGVATIQIVC
>seq_92
PMTVMAQFDNSRLIASSNTVASSGTLTIRNSARSQSSFTFKLLDAAGYRVAEIPHMAPGQKKSVSTEILAPGPYKIEALDEDTRRRSFLDICVVP
>seq_93
MVEINCFSDKGLIESDQPVVSENGHLTVHNDSAKGSQLDFLSLISNGVNINEAKNVYHGEYVEIPTLRWRPGEYRIHCRDVDSGKRFEHPLKVTE
>seq_94
YSVGLELDRDHYPVGSKATLKVNFSTEPEAPALVRVEIQSSDNTTKWQREFWMSKFTVMTAEFTVSECPGDYYIVVTVQFQDGSEEVANATATVN
>seq_95
FSPPSTMKSELLIAIISGRVDETTVKVRIEMDAAESTLTVKLLSEDGKEVKVETIKSVCTVAELEIDVSEVAEGNYEIVVTTGKGVSYVVTDRVR
>seq_96
DIHVTITDGIVTLDKKDVDTQPGEVRITLENKDGESKYVVITNTNEESVKINEEVGTGTHNYTLNLEDIGTYPLKFDFIAFDGTKNSATLTLTVS
>seq_97
TVSANVANGVVMFGSASHLTIAVDQPLIVKNLDDRFLGTFKLLCSDDTVVRELKKIRYNESASINLAPLTSGDYSLVAEFDKTHEENTLAITLVM
>seq_98
LALLCISDKEEYTYGETANDTVIWDAPMTMLAHINRVVLDQLGQQTYQQEFQAAASKTITSSHTVNFKAGDYLRELHVIKQDKELRRMTQEVTVS
>seq_99
KIRVAEESEYPRLELETGDEVQTSVKVCIEMTKDCDWKVSLVDANGKEISLLLARRVGINEPFSLDVMVCHLSHRYYKLVLENERDKLSLSIFVN
>seq_100
MVLNITCSPKTLIESDQPVRSENQPLTVHNIFAIISQADVLSLISNHVNINDAKNLYRNEYKEIPLYRIRPGEYTIHCRSVDSGKRRLTPLKVTE
>seq_101
LTVSVYVEISKAGDTWTLDVSVTVEANRPYTLDLVLTLFSEDGKNLREETFMIQSGDVKRAEFTSLEPGKLLLELQVLNERNQLIATEIKELQLK
>seq_102
SNVTLNLSKKEYFQGESATVTIKYSESTSYDVKYVLQVYSSQGELLREYQADDDTKTVDVAKFTVQYNKGNYRLYATAIFNDGNVLTATKDFRIS
>seq_103
PQIYQVTWKKGRLEISPGKIIPLASYFRFVNADDEEKDFVLYQQRGEEWISLYRVTNLKPGGETDLIDVRPLKSGTYKVLAKSPEKEYSLDIVIN
>seq_104
MNVNDPVIVITPQTKQVAIDDQVEFEITTIGSSPIKTIPFKDDHILSECEALHLEIGSDNHLIVKVDGAALEPGEYQVAGTHGDASEYAILILTG
>seq_105
SPPCTVRDVQVQWAETHLRVPTGRIVLTLANSDGGFAVQVTVLDESGRTIAKSQPTQAQAEVTVEFKDPGRYYVVCHVFRDGRQRAEGTLLIEKA
>seq_106
MSVYVVTVSGGRVFLDPGTTVPQNACLRFKNHSNTPVTAKLQLKVGPTKQSVKVWKNIGPASTTPLFRLVDLPKGEHIVRIESANSSVHGALTIE
>seq_107
SAEVHVKVVAGRVTITSGQKLGIAGTIIVENADAREIDLTLVDRVCKKNTSMKTFSGLKPGQRLPFGVKFKRSGGRVEITVSKEAGQDVAYIDVE
>seq_108
DCLTLLLEYETYAIGQELKLKDNLSKATQQESLNTLQILDIKGRVVQQLEGYSFGRQTMTVVITIQLPSGYRTFKSRVSYRDGSEFFSSSKFKVV
>seq_109
ANDVILTIKDGKVAVEEGTKLDAGDVIVLRNASNDFVDFSFWVKKTGTHTQIAEIKDAEPGTDHTVKIPRTTPAGEYIVRAANADHESEVNIILV
>seq_110
ELKDCILSVSVTKTKTVASTEFVVAYRSSSWTAVTVSVYDYFGRELKLIASCLVPVGEHQKEFTVDFSIYVAGRYMVKAELSDGDRSQASFEVEE
>seq_111
AALYTVTVKNGGLELKPGDLIARNNLLTFKNEDKNIRDFTLLRTEGNRYIDIFHWPNLRPGKSTEPYPLQDIPEGNYMVEATSMTFRSRLSIFVK
>seq_112
GTLYLQTEKTVFKQDEVVHLRALLDAAPAEAGEVTLRILNDKGEVVKKLTSSLNSFNALKFDFRVRHRQGNYRVVIEMTFPDGRKLTRELTIEVE
>seq_113
LEHVEVVRDWLYAGDRRRLTIDKMDDLLVSTGNAQEDWDLLLLDMCGNSLAVKQLIAPAAEGKLALSDLPVGNYTLQLINLSSGKTSTLEVILRG
>seq_114
GDTTIIVIENGKLLVEKGHSTTPGGKIVVINKNDILVDVKIYHRHGNKWGLIAEVKNLVPGVEVEIVVPEELQPGQYRIFAENHLFESYAYVALL
>seq_115
SAISLVADRRIYVQGESAQLTVILEEAPVEPALVRITVLDRQDRECKKMTSMMNKSRSKVWNFTPFFDKGYYHVAVQVTTPSKETLQDCKLIIIK
>seq_116
IKVRLELDREFYTKGSLLDIKIEFSQPPSGNVDVFLKILTQGGDTLNWQKFVSQKQTEKIIHLTVEEEDGFYTTQVIVEYPDGSVETTQTTFKVS
>seq_117
GTLYLQTEKKVFGQDEVVHLRALLDAAPAEAGEVTLRLEDRKGRVVKKLTSSLNSFNALKFDFRVRHRQGNYRVTITMTDPDGRKLHRALTIEVE
>seq_118
NWSLEISVTEKGKIFKNHDLVSISQLVTGIFFHNMLRIQLLAENGGLVRVLSTHQFNKSLEQIVTLQDFAAGSYKACGSSSIGLKKEVPLLIRGG
>seq_119
SQRIVVEDGLLSIDNQTHCQLPASTRLQLVNAEQSQARTFRLEDAQGSVCRDIPTIADGGSAALLGQESLPPKQYHVRATQSGGQVDQLALNVTH
>seq_120
MWSVQIEHRDGRIEPTGGTHIPEIGVLGIRNMGKEPVDLTISDEVGTQKFTKDTHKPVRPEEMIDFPLQFLQTSGKIVVQARDELGKGTDSILVK
>seq_121
ATDLDLKVLFHPAEINRENHAEYIVESSDPDRQGTTALAVLDVSDQVLHKSFVDVRRGEGSVSVNEREFHAGLYTLVATTLGASDAETDVKILNK
>seq_122
VEITISVLNETHKKGEKVSVTITLSNALPMNSYIKFEIRDHHGKVLYEKIFMHSAKQTDTIFLFNFQPNGEYTVQAIEILKEGYEFTEKLSLKVK
>seq_123
VKLRVIGGSNIRAEVEKEYYNDFISLTITSIMETKQVRVIVENREGKELRQVCESEIPANSKKVWTIDISEYQDGDYSLTVEAGNETHIIEFFQY
>seq_124
HKFDLRVDKEHYKKGSDVALRVSKLGERSHWKKVSLNILDKDCVELYKREHEANTAQSLELTFKVEIDPGEVTVQVMETHDDGKIANAVKQITIN
>seq_125
HQTYTLTVENQRIRVTPGDTINTESFLTFENKGGEVHNFTLTQTVGESNKILYEVDEMKPNEKSALIDITAIPLGKYFIAFKNTTYNAKYTLTVS
>seq_126
LNVTLNLDTSSYERNTLLKVEILFSKPPQQDAKVEIRVLDSEGRVVSTECTCLQKRVSIKYSLMVTYPPGDYRLEAEVHFQDGSTKQASSLFTVK
>seq_127
HNFVTLVETDKATYSAREVLRVRAVNSGSRAPNNVEISLYDPEQVELKSTDATVLDPNSIVEFLFDLSKLKEGYYEVRAMVDGKNKLSKKVETTQ
>seq_128
HHHNTLTNEYVRVELIDGDTVNTEVRVTIETKENESIVVTLTNANNETLKINYEGDIEAKAPHELLIDISALPSGKYFIVVKNTKDTVKLTLTLS
>seq_129
ALEIEVVDGQLRYGDRDQLQIDARQPLRVVNQEESKVHAVVLLDADGREVAAWPIIPPKARVSYERLSDLERGQRMLHAHGDDQAEALVKLHVSL
>seq_130
ALRVTIHLDAGRLSTTAITVAEHAKLQIENASNKRKPVDFRLIAADGEVRYSIPELGYGEKHVICMDEFEVGDYELVARALDTAVIGKAMLRIES
>seq_131
MAIPDGSALHIDAKDGRNELDETDYIFVNLGHDEYPITTEVHSENGELLKQLVNVQAPRHFHGYIQLSSLGEGQLRIYAGGDNQSKQMEITVVAP
>seq_132
AALDLTLDKSKYGRGEKVKLTATLNLRLEHEGDLNIEILDQNNTELNRYEDKINARSKRVKVYEVSEPLGRYDVVAALTTNDGMTFESRVSIFVF
>seq_133
TPCEVQDINVPPGEIVEVRFSQDDIMRIDELSLKADEVLRSEFQQQAPGVYKSELKEGEKVLFELTCCGVAEIRTTYQSVTLTGAGARVVLCADP
>seq_134
GPVVTLTKGQLGPGSRQIIQVPGTINLDVSNRDAETAILELKDNATGQPVLGPFTLGPAASSLRFDLSFLKPGRYTLKAIFADGKHAQLYLNVFE
>seq_135
MEVHLTTDRDLYSLGIELLIIVQLSDPPTSDVQNLATVLDEKDNVVLEQSGLIYTETVLQFSRTLNAKNGVYRYKVTVTLPDGTTYEASTKITVK
>seq_136
TVVNLTLSSEDFPAGETLNITISTDKEFEAPADVKVVVLNQNREVLKNISMDLEMKKQIKISEIVDIPTGNVLMMITVTTPTGKKYQAEGLFNIG
>seq_137
NVEIQCDSEKGLIRSTQPVASENSPLTVHNDSAIISQLDFLSLISNGVNQNELKNVYQGEYKEFDLTRIRPGDYTIHCIEVDSGKRRLHPFKVTQ
>seq_138
HFVIEISNRVVQCGRQKYLQVPSSEVLKIKNRGNNAEFNYELLGLTGDVLEAQLSLVPDVMNAIDTRGIEPGDYMDVATDAETGEQFNGKIVVKT
>seq_139
SATFTINIDTNSITASSTTVPSTGHVKVVNASPKDYVIDRRLLDASGHTCREIKDLKKGQSADLDLHDFKAGEYVLEATNTYSRQTASFDVKVVS
>seq_140
NVEIQCKSGKLLKGEDQPVRVENSHPLKVNTFNYISQLDFLSLISNNVNQNEAKNVYPNEYTEFDLTRIRPGEYRIHCREVDSGKRHLCTLEVYQ
>seq_141
FLGQKYRPVSVAVRPDGVITDDTFKVTVQLFREGTICVRVLDEDGRVLRRLHFPKEGRYGIRLLSEDGRPIPEGRYTVEVNNGQHAEALSPTVKR
>seq_142
DHLNGLLDSTHFVIGNDVRCTIQASELQKNFSLLILQGLSLEQRLLWQEQKLLYAQKNLNFYFQIDFEWGTYIVVKTQTEANGQGFTASQVIQIR
>seq_143
STLKLAAAPVVAVGQEGKAEFDLKTPFEVINYDYSLDGKKLGANPAGQAAADIKNRAATAGVFSSAADAPQSVTATAVTEPRKLTASAAITVLAP
>seq_144
SITIETVNNINGIGNSDTLHISLGDKVELINYDETKVIAREKVMESGEVCREIPEIAPRASRILEGNLFLKEGEYKIIATTYDRQVASFELHIYS
>seq_145
GSNPSLVSGNLSITISNGITISKKVKVTITLDEPRNKIVVELLNSEAQVELKVEIENVPKNFEYDFDISFIDSGDYLCTVTDEDGTKATVSVTID
>seq_146
AVATLQLERNNYKRGSRVLFVLLLDAPPNEDAKVVITVLDQDGNEVWSDSCNLKKTQKHVFSRICKASEGRYKVIVVVTFPQGYELTAEVSIVVG
>seq_147
LEHVEVVRDWLYAGDRRRLTIDKMDDLLVSTGGAQEDWDLLLLDMCGNSLAVKQLIAPAAEGKLALSDLPVGNYTLQLINLSSGKTSTLEVILRG
>seq_148
MTLTLELAREQYNRGETVTATLTLAFEPEEDHLVRFEIKRTDETLIYSSESVLNKERAMKHDFVANHPRGYFILKAIVIASDRRMFEAKRSFRII
>seq_149
HFLRIETERSEYQKREIAKVTISLEAPLPDHAALLLELYNEKNGLVKQKSLAQNCKVSRVMNFEVTEEGGTYILRAVLTTAEGQRQFAQKRVVVT
>seq_150
HEAHLTTDRDLYSMGIELLIIVQLSGPDTHWVQNLATLLDEKGNVVLEQEFDIFTGNVLSMSRTLNAKNGVYTYKVTLTTAEGKTYEASTKITVK
>seq_151
HHHNTSTNEGLTVSVTNGDTQNTEVDLTIETKEEKSKVTVRVLASDESVKINKEIDNVPKAFTLTIDLSTWPPGKYDIIATDETGTTVKVTLTLS
>seq_152
KFLKKELDRNRYNIGSKAKVTVTFSHPLPNTAKVELKILAKDGIVKDRLSREAKREETLNWDFVMKMPTGWYRIQAKRFTDNKNTIVCEDTFWVM
>seq_153
KKVVISLNRTNFTRGQPIVMYCNHEEADPEPTNLTVAVLDSKGRVIDEFSTESDQRAVQNVTIVNRWPDGMRNLQVWKTSPDGSKEVASVKVNIN
>seq_154
SNVATWVISDGKATLTSGNAITPDGTTIVPNHSGDHVDVVWKLREVDKKRTLQQQNDLLPGQNYSLRIPEATAPGLYTLTISDNDYKSECMFRLL
>seq_155
GDTTIVTYENKKLLVEKGHYIALEDKIRFINKSDILVDYKLYQRHGNKWGIIFEIKNLVPGAETSDFDVAELQPGQYRFFAENHEHESYSYLVLG
>seq_156
AVMMTITPPVGDLSTTPGATAEHVLQLQFGLSASGKYTVQLLDAQGQQLRELYNGRMPAGKHVFQIDESQFAPGEYVLWASDGAVTGKLMARIES
>seq_157
HRFVKIVIKRGELSIEEGISVQPNDHIEIRNEDNQNVTVQILLKVGNDYMVLKRADNVAFGTELQFQIPKETPPGVYMVRATSELFSARKEFLLG
>seq_158
EAKVEVMVKDGKLFIVNGTEVPPGGLINIFNEGKQPATITLELLQVGWASPLRTITNLLDKEVSSIAVRAGLAPGTYLVTASNSTFNDVTEVRLH
>seq_159
LEHVEVVRDWLYVGDRRRLTVDKMADLLVENIGNQEGWDLLLLDMCGNSLAVKQDIAPAAEGKLALSDLPVGNYTLQLINLSSGKTSTLELILRG
>seq_160
ANECEVYIKDGWLSITKGKEVVRGGYIVVRNQSRQKTDVRLLPSEGDVKKLVKYLKNLDPGQVAKLQIPINAKDGKYVLLSESADFSDEVDFYLS
>seq_161
MDFPTWQFGDLSLQLLGGDVFSKKVRVGISLDQEQQLITVQVVNADGKIVYTETINGASVIQEIFIDIQDWQPGDYKLKATIEKGQVVQLWFQVY
>seq_162
LTVSIKTDKEVYSRDEKIEFTITFQRPPATVCSVTLLIRDKEEEVLYSESLDANKKTSITYSYEANFPGGLYTLLCIVTFSDGSEERGSREIKIE
>seq_163
LTYSWRNNDWLYIQIWSGDSVSGKVKVRIYMLDDVNYECILLDKNGNRVAVMKTETYFANDEEILYLTVSLLMSGQYVLRVNTDIDTLDLPLQVK
>seq_164
NTRVEVFLIKVIENGEVVNEKVVQVPLGQTAEFIVNSPIGTTHTTFLVFDAQQQIVLEFKPLKVGQTQNLQATNVSISVTALDYHTMRVSVTVNE
>seq_165
AALDLTLDKSKYGRGEKVKLTATLNLRPEHEGDLNIEILDQNNTELNRYSDKINARSKRVKVYEMSEPLGRYDVVAAMTENDGMTFESRVSIFVF
>seq_166
MLHELILNRNIYVRGEHVTLKLILSAAPQSEAQVTILVLSSKGKTKFSNHFTLKKQKEFTNIFTARYPAWEYRVECTVTLANGQEYKANIQITIK
>seq_167
MVRCTVQQGKVVFDEDSVSVEPGKVHFTVTNIDVSTLVSLEVLDENGNVVAKSHAALNSASLEVELTKPGKYTVVCIAYQTGVQVDQGKAFLEVA
>seq_168
DSTVTLTVEDDRVSVTQGDSLNTEGFVTIENKGGESHDVTLTWTNGESKKILKEIGDMKKNETYTLELIFPKPGGKLDIAFKNTKGTGKATLDVS
>seq_169
MDITLTLDKELYKSGEKMTITVDLDKETPYPATLEVRVLTVDGLLPKECSLFDDCHLSTQHEMNVEFQAGQFTVCAILRLPDCTVREASRLVAVE
>seq_170
MLHELILNRNIYVRGEHVTLKLILSAAPQSEAQVTILVLSSKGKTKFSNHFTLKKQKEFTNIFTVRLPAWEYRVECTVTLANGQEYKANIQITVK
>seq_171
GMNDSQMNAGLSISLVNGSVHKKTLELEITLQKPVKQTKVQLKNANGKTVKSELHYNVPTSFRLIIDISNLATGKYKIELSTADGTTQSVSAEIQ
>seq_172
RATARVTISNDTLRIDPGTQLAQNGTLTIHNASDRAHNFEIYQRQGDVDLLVYQQKNLEPGQETPPINLSYLLAGPYRIQVRGESFTDQLDFLVL
>seq_173
NLYEVTIATIAVGPDYVLVKVYKDGDKIKIFIDADDAREELTVEIWNSKNIVLHKTFLDPYGTKRISLKNLGTTLTIRISSSDYCRTYEI
>seq_174
SILTIATRKDNNISIHPGNEVLQNDFIQFENKSTAKRTFRLLKKAGQRWELAKLWSGLKPGQSSPPYNTTYLPLGDYRLIVETSAYNVAVRITIG
>seq_175
KATVNVTEGHVSFGQNSRLSINHHYSFFVQTKGHPNTITVELEDEQGLVVKSIPTQQPKYQSKLDTKELEPGKCLLTVKDADTSSSHVSEVTLFW
>seq_176
KIRVAEESQYPRLELETGDEVQTSVKLCIEMTKDCDWKVSLVDANGKEISLLLARRVGINEPFSLDVMVCHLSHRYYKLVLENEYDKLSLSIFVN
>seq_177
MRLVLKLERSVYRKGEYVTFTVSASMPWREPGELIFTVKDKAGNVLHEQSFTFDQKKHFTVRVKANFEDGDYDVQAEATWPDFRYVLAHADVRVN
>seq_178
LELKLELEKTEYHRGETFKGTLQLDAPPSRVSLVRVTVLNPKGEVVREKELLLAGQSQLEFSFRQNAEAGAYWIRQEVQLLDGKILSAEVDCKVT
>seq_179
GRVFRVSQGSFQVGSQTHITVDKYKPLHLETDSWQLKLKVLLLDENGQVRMKVEDITPGSGFQIDLKVLEPGNWTLHAEEMSTGRVSKANHKLYA
>seq_180
DDIALTTCRDSYREGDTAWIEVSDVSPLPNTAELTLRVKNEAGELEFAKQHYLGKRARYQWQWGVHLPPGRYTVVAVVTDINERKWEAEVHIQVL
>seq_181
HKRAVVVIEDGKLAIKAGSILPKKGYVEIENASDEMHEVCVDRRSPVDMVETLYQITMNPEEKLVIPFEFPEAGNKLIIIMKNEANVDTAEITVV
>seq_182
NVITLETEKMDYMLGDTLILNVSFSSAPESDCRLHVSLLDENQEEREKWVVVTGGKQGVQIEKEVKFEPGLYKIRVVNTAPSGTSFEDNTDIAIL
>seq_183
MVVTVAAERDEYALKEAIKLTVALEEEAEKDAQVEFTVKSSKGAELYTMTTRLNKAKKQEYVFEVKQKPGGYVVEVTVNFQSGIKKSAKKTFKVV
>seq_184
AKVVSVSLAEGLLSLIIGKAIEPGEKLVIKNESFKVHHINVHLVVGDKKILVREIKRVFPGSTEVVKLPPYLQPGTYVVEATNQDFSASVKVGLQ
>seq_185
ATIELHLDRDAYEQGEMVELTVVLSAAPSSDAVLNLRCLDYKGVMWFGSEWKMQKQEKQVYTFTINLPNGHYLLTAWVTEENGKQLEASVRFTIY
>seq_186
SFTPVLPKAHLHLSANGVYFTEQNLNITEVTPREAEGTLHVIPSDGAVVEEKPLQKEKKRDCEIDLSNLQPGAYDILAPLRDGQTACKRFQVRHI
>seq_187
SNVVVLENGELGSGPTKAVTVSKDATLSVLNKEPRAVEVRLEDVATGAGVIGPFLLKPAGAQCDMDLDNPEDGEYRIVATLADGARAKADVTVVA
>seq_188
RSTVRLEISNGTVRITSGTQLAPGGTLLIHNRDDQAHNFEIYTRQGDVDLLQAQQKNLQPGQTVSLSVPDDTLAGLYRLQVRGESFTDQAEFLVL
>seq_189
EAVATFKIADGKVDIVGFATVGDHFTIKVTNDSEKAYKIQLFEIKPDGERNLLHQTELSPNETKVFDVDLAKSGGIILLEASSSEHHDKGELYVK
>seq_190
MVVTVAAERDEYALKEAIKLTVALEEEAEKDAQVEFTVKSSKGAELYTMTTRLNKAKKQEYVFEVKQKPGGYVVEVTVNFQSGIKKTAKKTFKVV
>seq_191
SAISLVADRRIYVQGESAQLTVILEEAPVEPALVRITVLDRQDRECKKMTSMLNKSRSKVWNFTPFFDKGYYIVRVEVTTPSKETLQDCKLIIIK
>seq_192
HHVQTLTGTIVRVRVKDDIYQNTVVLNVEATFGEESIVVTLTNTNRESLKILAETGIKAKATKELRIDITALPSGKYFIEVKSTKNVNIYTLTVS
>seq_193
MNLTITLDSEEHEAGEIVRVDITLDKAPPENDAIRVTVLSPEGKVLRSEEFVGDRKSKCSVSFVEDFPEGVYTIRCELTFPSGKVVSYETTDRVR
>seq_194
ATIELHLDRDAYEQGEMVELTVVLSAAPSSDAVLNIRILDYKGNMIFGSEWKMQKQEKQVYTFTINCANGHYLVTAWVTEPNGKQLEASVRFTIY
>seq_195
AVATLQLERTNYKRGSRVVFVLLLDAPPNEDAKVVITVLDQDGNEVWSDSANLKKTQKHVFSRICKASPGRYTVRVVVTFPQGYELTAEVSIVVE
>seq_196
KSTVIIEINDGELKNISGEVADSQFEIVIENKSKKAFNIKVIFAMPGGVQKVLMETKIKPGQSKTSKITVNVTGGILEIRGKHDEHSSRTMILIN
>seq_197
LTVSLKTDKEVYSRDEKIEFTITFQRPPATVCSVTLLIRDKEEEVLYSESLDANKKTSITYSYEADFPGGLYTLLCIVTFSDGSEERGSREIKIE
>seq_198
LTYSWRNNDWIYIQIWSGDSVSNKVKVRIYMLDDEHIECELLDKNGNRVAVMYEGYYFANDEEILYLTVSLLMSGQYVLRVNTDIDTLDLPLQVK
>seq_199
MNVPPRVIVVPPQTKVILRADQPIKVRVDRLSPDKKTIPGSVDHDLSPGEYQWEAEGNSVYEITLLSEGTLELETHQRAHTLNGEGEKLIIIAEG

"""

print(f"Ready to fold sequences. Total input length: {len(fasta_input)} characters")

Ready to fold sequences. Total input length: 20897 characters


In [ ]:
%%time
#@title **Fold all sequences**

from string import ascii_uppercase, ascii_lowercase
import hashlib, re, os
import numpy as np
import torch
from jax.tree_util import tree_map
from scipy.special import softmax
import gc

def parse_output(output):
  pae = (output["aligned_confidence_probs"][0] * np.arange(64)).mean(-1) * 31
  plddt = output["plddt"][0,:,1]
  bins = np.append(0,np.linspace(2.3125,21.6875,63))
  sm_contacts = softmax(output["distogram_logits"],-1)[0]
  sm_contacts = sm_contacts[...,bins<8].sum(-1)
  xyz = output["positions"][-1,0,:,1]
  mask = output["atom37_atom_exists"][0,:,1] == 1
  o = {"pae":pae[mask,:][:,mask],
       "plddt":plddt[mask],
       "sm_contacts":sm_contacts[mask,:][:,mask],
       "xyz":xyz[mask]}
  return o

def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()
alphabet_list = list(ascii_uppercase+ascii_lowercase)

# Parse FASTA
sequences = {}
current_name = None
for line in fasta_input.strip().split('\n'):
    if line.startswith('>'):
        current_name = line[1:].split()[0]  # Get name until first space
        sequences[current_name] = ""
    elif current_name:
        sequences[current_name] += line.strip()

print(f"Found {len(sequences)} sequences to fold")
print("-" * 60)

num_recycles = 3  #@param ["0", "1", "2", "3", "6", "12", "24"] {type:"raw"}
chain_linker = 25
jobname = "esmfold_batch"

# Load model once (for efficiency)
if "model" not in dir() or model_name != model_name_:
  if "model" in dir():
    del model
    gc.collect()
    if torch.cuda.is_available():
      torch.cuda.empty_cache()

  model = torch.load(model_name, weights_only=False)
  model.eval().cuda().requires_grad_(False)
  model_name_ = model_name
  print("Model loaded")

# ====== BATCH FOLDING ======
results_summary = []
os.system(f"mkdir -p {jobname}")

for idx, (seq_name, sequence) in enumerate(sequences.items(), 1):
    print(f"\n[{idx}/{len(sequences)}] Folding {seq_name}...")

    # Clean sequence
    sequence = re.sub("[^A-Z:]", "", sequence.replace("/",":").upper())
    sequence = re.sub(":+",":",sequence)
    sequence = re.sub("^[:]+","",sequence)
    sequence = re.sub("[:]+$","",sequence)

    seqs = sequence.split(":")
    lengths = [len(s) for s in seqs]
    length = sum(lengths)

    if length > 900:
        print(f"  ⚠ WARNING: Sequence too long ({length} aa), skipping")
        results_summary.append({
            'name': seq_name,
            'length': length,
            'ptm': None,
            'plddt': None,
            'status': 'SKIPPED (too long)'
        })
        continue

    # Set chunk size based on length
    if length > 700:
        model.set_chunk_size(64)
    else:
        model.set_chunk_size(128)

    torch.cuda.empty_cache()

    try:
        output = model.infer(sequence,
                           num_recycles=num_recycles,
                           chain_linker="X"*chain_linker,
                           residue_index_offset=512)

        pdb_str = model.output_to_pdb(output)[0]
        output = tree_map(lambda x: x.cpu().numpy(), output)
        ptm = output["ptm"][0]
        plddt = output["plddt"][0,...,1].mean()
        O = parse_output(output)

        print(f"  ✓ ptm: {ptm:.3f} | plddt: {plddt:.3f}")

        # Save outputs
        prefix = f"{jobname}/{seq_name}_ptm{ptm:.3f}"
        np.savetxt(f"{prefix}.pae.txt", O["pae"], "%.3f")
        with open(f"{prefix}.pdb", "w") as out:
            out.write(pdb_str)

        results_summary.append({
            'name': seq_name,
            'length': length,
            'ptm': f"{ptm:.3f}",
            'plddt': f"{plddt:.3f}",
            'status': 'OK'
        })

    except Exception as e:
        print(f"  ✗ ERROR: {str(e)[:50]}")
        results_summary.append({
            'name': seq_name,
            'length': length,
            'ptm': None,
            'plddt': None,
            'status': f'ERROR: {str(e)[:30]}'
        })

# Summary
print("\n" + "="*70)
print("FOLDING COMPLETE")
print("="*70)
import pandas as pd
df = pd.DataFrame(results_summary)
print(df.to_string(index=False))
df.to_csv(f"{jobname}/summary.csv", index=False)
print(f"\n✓ Results saved to: {jobname}/")
print(f"✓ Summary: {jobname}/summary.csv")

Found 200 sequences to fold
------------------------------------------------------------
Model loaded

[1/200] Folding seq_0...
  ✓ ptm: 0.361 | plddt: 47.800

[2/200] Folding seq_1...
  ✓ ptm: 0.771 | plddt: 84.224

[3/200] Folding seq_2...
  ✓ ptm: 0.615 | plddt: 71.583

[4/200] Folding seq_3...
  ✓ ptm: 0.535 | plddt: 65.770

[5/200] Folding seq_4...
  ✓ ptm: 0.686 | plddt: 79.067

[6/200] Folding seq_5...
  ✓ ptm: 0.764 | plddt: 82.289

[7/200] Folding seq_6...
  ✓ ptm: 0.582 | plddt: 66.444

[8/200] Folding seq_7...
  ✓ ptm: 0.612 | plddt: 77.936

[9/200] Folding seq_8...
  ✓ ptm: 0.529 | plddt: 63.450

[10/200] Folding seq_9...
  ✓ ptm: 0.723 | plddt: 78.480

[11/200] Folding seq_10...
  ✓ ptm: 0.761 | plddt: 81.659

[12/200] Folding seq_11...
  ✓ ptm: 0.580 | plddt: 71.325

[13/200] Folding seq_12...
  ✓ ptm: 0.521 | plddt: 68.102

[14/200] Folding seq_13...
  ✓ ptm: 0.751 | plddt: 81.070

[15/200] Folding seq_14...
  ✓ ptm: 0.631 | plddt: 72.866

[16/200] Folding seq_15...
  ✓ 

In [ ]:
#@title **Download Results**
from google.colab import files
import os

jobname = "esmfold_batch"

if os.path.isdir(jobname):
    os.system(f"cd {jobname} && ls -lh")
    print(f"\nZipping {jobname}...")
    os.system(f"zip -r -q {jobname}.zip {jobname}/")
    print(f"Downloading {jobname}.zip...")
    files.download(f'{jobname}.zip')
    print("✓ Download started!")
else:
    print(f"Error: {jobname}/ folder not found. Run the Fold cell first.")

## Next Steps

After downloading your PDB files:

1. **Extract the zip** locally
2. **Run FoldSeek locally** (or on the web):
   ```bash
   foldseek search *.pdb cath_database results
   ```
3. **View results** to validate fold classification